# MNIST Autoencoder — Single Student Binder Test

This notebook simulates **one student's workload** in a Binder session.

It will:
- load MNIST from TensorFlow/Keras,
- train a small convolutional autoencoder,
- evaluate reconstruction quality,
- monitor RAM usage during the run,
- record training time and total notebook runtime,
- save RAM measurements to `binder_ram_usage.csv`.

> **Note:** An autoencoder reconstructs images; it does not classify digits. The `binary_accuracy` reported here is **pixel reconstruction accuracy**, while **MSE** and **MAE** are the more meaningful metrics.

In [ ]:
# Optional: install psutil if your Binder image does not already include it
# !pip install -q psutil


## 1. Imports and session information

In [ ]:
import os
import time
import threading
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('PID:', os.getpid())
print('Session started:', datetime.now().isoformat(timespec='seconds'))


## 2. Start RAM monitoring

In [ ]:
ram_log = []
stop_monitor = False
session_start = time.perf_counter()

def get_total_ram_mb():
    process = psutil.Process(os.getpid())
    total = process.memory_info().rss

    # Include child processes if TensorFlow creates any
    for child in process.children(recursive=True):
        try:
            total += child.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    return total / (1024 ** 2)

def monitor_ram(interval=1.0):
    global stop_monitor
    while not stop_monitor:
        try:
            ram_mb = get_total_ram_mb()
            elapsed = time.perf_counter() - session_start
            ram_log.append({
                'elapsed_sec': elapsed,
                'ram_mb': ram_mb
            })
        except Exception:
            pass
        time.sleep(interval)

monitor_thread = threading.Thread(target=monitor_ram, daemon=True)
monitor_thread.start()

print('RAM monitoring started.')
print(f'Initial RAM: {get_total_ram_mb():.1f} MB')


## 3. Load and prepare MNIST

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Keep the workload modest for a ~2 GB Binder session
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 2000

x_train = x_train[:TRAIN_SAMPLES].astype('float32') / 255.0
x_test = x_test[:TEST_SAMPLES].astype('float32') / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print('Training shape:', x_train.shape)
print('Testing shape :', x_test.shape)
print(f'RAM after loading MNIST: {get_total_ram_mb():.1f} MB')


## 4. Show sample images

In [ ]:
plt.figure(figsize=(10, 2))
for i in range(10):
    ax = plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.suptitle('Sample MNIST Images')
plt.show()


## 5. Build the convolutional autoencoder

In [ ]:
inputs = tf.keras.Input(shape=(28, 28, 1))

# Encoder
x = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling2D(2, padding='same')(x)
x = tf.keras.layers.Conv2D(8, 3, activation='relu', padding='same')(x)
encoded = tf.keras.layers.MaxPooling2D(2, padding='same', name='encoded')(x)

# Decoder
x = tf.keras.layers.Conv2D(8, 3, activation='relu', padding='same')(encoded)
x = tf.keras.layers.UpSampling2D(2)(x)
x = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.UpSampling2D(2)(x)
outputs = tf.keras.layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

autoencoder = tf.keras.Model(inputs, outputs)

autoencoder.compile(
    optimizer='adam',
    loss='mse',
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name='mae'),
        tf.keras.metrics.BinaryAccuracy(name='binary_accuracy', threshold=0.5)
    ]
)

autoencoder.summary()


## 6. Train and time the model

In [ ]:
EPOCHS = 1
BATCH_SIZE = 128

training_start = time.perf_counter()

history = autoencoder.fit(
    x_train,
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1
)

training_time = time.perf_counter() - training_start

print(f'\nTraining time: {training_time:.2f} seconds')
print(f'RAM after training: {get_total_ram_mb():.1f} MB')


## 7. Evaluate reconstruction quality

In [ ]:
metrics = autoencoder.evaluate(
    x_test,
    x_test,
    batch_size=BATCH_SIZE,
    verbose=0,
    return_dict=True
)

print(f"MSE loss              : {metrics['loss']:.6f}")
print(f"MAE                   : {metrics['mae']:.6f}")
print(f"Pixel binary accuracy : {metrics['binary_accuracy'] * 100:.2f}%")


## 8. Display original vs reconstructed images

In [ ]:
reconstructed = autoencoder.predict(x_test[:10], verbose=0)

plt.figure(figsize=(12, 4))

for i in range(10):
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0:
        ax.set_title('Original', fontsize=9)

    ax = plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0:
        ax.set_title('Rebuilt', fontsize=9)

plt.tight_layout()
plt.show()


## 9. Stop RAM monitoring and save results

In [ ]:
stop_monitor = True
monitor_thread.join(timeout=3)

ram_df = pd.DataFrame(ram_log)

peak_ram = ram_df['ram_mb'].max() if len(ram_df) else float('nan')
avg_ram = ram_df['ram_mb'].mean() if len(ram_df) else float('nan')
total_time = time.perf_counter() - session_start

ram_df.to_csv('binder_ram_usage.csv', index=False)

print('RAM data saved to binder_ram_usage.csv')


## 10. Plot RAM usage

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ram_df['elapsed_sec'], ram_df['ram_mb'])
plt.xlabel('Elapsed time (seconds)')
plt.ylabel('RAM usage (MB)')
plt.title('Binder RAM Usage During MNIST Autoencoder Test')
plt.grid(True)
plt.show()


## 11. Final student session report

In [ ]:
print('\n' + '=' * 60)
print('SINGLE-STUDENT BINDER TEST REPORT')
print('=' * 60)
print('Training completed     : YES')
print(f'Training time          : {training_time:.2f} seconds')
print(f'Total notebook runtime : {total_time:.2f} seconds')
print(f'Peak RAM usage         : {peak_ram:.1f} MB')
print(f'Average RAM usage      : {avg_ram:.1f} MB')
print(f'MSE                    : {metrics["loss"]:.6f}')
print(f'MAE                    : {metrics["mae"]:.6f}')
print(f'Pixel accuracy         : {metrics["binary_accuracy"] * 100:.2f}%')
print('Crash detected         : NO (notebook reached final cell)')
print('=' * 60)
